# Lab 03: Building a Retrieval Agent

## Overview

In this lab you will build a **Retrieval-Augmented Generation (RAG) agent** that uses a Databricks Vector Search index as a retrieval tool, orchestrates responses with a LangChain agent, logs the model with MLflow, and registers it in Unity Catalog.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Application Development (30%) |
| **Key Skills** | Vector Search retrieval, LangChain tool calling, MLflow model logging, UC registration |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | $1–2 |

### What You Will Build

```
User Query
    └─► Retrieval Tool (Vector Search)
            └─► LangChain Tool-Calling Agent
                    └─► ChatDatabricks LLM
                            └─► Grounded Response
```

The finished agent is logged as an MLflow model and promoted to the Unity Catalog model registry so it can be deployed via Model Serving.

## Architecture Diagram

![Architecture Diagram](../assets/diagrams/lab-03-building-retrieval-agent.png)

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph --quiet
dbutils.library.restartPython()

In [ ]:
# Configuration
CATALOG = "genai_lab_guide"
SCHEMA = "default"

# LLM for agent tasks (Llama 4 has reliable tool calling)
LLM_ENDPOINT = "databricks-llama-4-maverick"
VS_ENDPOINT  = "genai_lab_guide_vs_endpoint"  # Vector Search endpoint name
VS_INDEX     = f"{CATALOG}.{SCHEMA}.arxiv_chunks_index"  # Fully-qualified index name

print(f"Catalog   : {CATALOG}")
print(f"Schema    : {SCHEMA}")
print(f"VS Index  : {VS_INDEX}")

## A. Build a Retrieval Tool

The retrieval tool wraps a call to the Databricks Vector Search index.

- **`similarity_search`** supports both dense (embedding) and `hybrid` (dense + sparse BM25) retrieval modes.  
- The function returns a single formatted string so that any LLM can consume the context directly.  
- Keeping the tool output as plain text avoids serialisation issues when LangChain passes context across agent steps.

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc   = VectorSearchClient()
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)


def retrieve_context(query: str, num_results: int = 3) -> str:
    """Query the Vector Search index and return formatted context passages."""
    results = index.similarity_search(
        query_text=query,
        columns=["chunk_text", "path"],
        num_results=num_results,
        query_type="HYBRID",
    )
    docs = results.get("result", {}).get("data_array", [])

    context_parts = []
    for doc in docs:
        source = doc[1].split("/")[-1].replace(".pdf", "")
        context_parts.append(f"[Source: {source}]\n{doc[0]}")

    return "\n\n---\n\n".join(context_parts)


# Smoke-test the retrieval function
sample = retrieve_context("transformer architecture")
print(sample[:500])

## B. Create the RAG Agent

We use the **LangGraph ReAct agent** pattern:

1. Wrap the retrieval function as a `@tool` so the LLM can decide when to invoke it.
2. Provide a system prompt string that instructs the model to always retrieve before answering.
3. `create_react_agent` binds the tool to the LLM and manages the reasoning loop.


In [ ]:
from langchain_community.chat_models import ChatDatabricks
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ── 1. LLM ──────────────────────────────────────────────────────────────────
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    max_tokens=1024,
    temperature=0.1,
)

# ── 2. Retrieval tool ────────────────────────────────────────────────────────
@tool
def search_arxiv_papers(query: str) -> str:
    """Search the arXiv paper corpus for context relevant to the user's question.
    Always call this tool before answering research questions."""
    return retrieve_context(query)

tools = [search_arxiv_papers]

# ── 3. System prompt (plain string — langgraph style) ────────────────────────
SYSTEM_PROMPT = (
    "You are an arXiv Research Assistant. "
    "Always use the search_arxiv_papers tool to retrieve relevant context "
    "before answering. Ground every answer in the retrieved passages and "
    "cite the source identifiers provided in brackets."
)

# ── 4. Agent (langgraph ReAct) ───────────────────────────────────────────────
agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

print("Agent ready.")


## C. Test the Agent

Run two representative questions to verify the retrieval-to-response pipeline end-to-end.

In [ ]:
test_queries = [
    "How does the attention mechanism work?",
    "What is LoRA and why is it useful for fine-tuning?",
]

for query in test_queries:
    print("=" * 70)
    print(f"Q: {query}")
    print("-" * 70)
    response = agent.invoke({"messages": [{"role": "user", "content": query}]})
    print(f'A: {response["messages"][-1].content}')
    print()


## D. Log with MLflow

Logging the agent as an MLflow model captures:

- The **model signature** (input/output schema) inferred from a sample run.  
- The LangChain chain definition, tool definitions, and all pip dependencies.  
- A run-level record of the experiment, enabling comparison across agent versions.

In [ ]:
import mlflow

username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/genai-lab-guide/lab-03-retrieval-agent")

with mlflow.start_run(run_name="arxiv-rag-agent-v1") as run:
    mlflow.log_params({
        "llm_endpoint"  : LLM_ENDPOINT,
        "retrieval_k"   : 3,
        "query_type"    : "hybrid",
        "vs_index"      : VS_INDEX,
    })
    # Note: In LangChain v1+, model logging uses models-from-code or ChatAgent.
    # We log params and traces here; full model registration happens in Lab 05
    # with the ChatAgent wrapper.
    print(f"Run ID: {run.info.run_id}")
    print("Params logged. Model will be registered as ChatAgent in Lab 05.")


## E. Register in Unity Catalog

Registering the model in Unity Catalog makes it available for:

- **Model Serving** — deploy as a REST endpoint with one click.  
- **Lineage tracking** — UC records which data assets the model depends on.  
- **Governance** — apply column masks, row filters, and access controls at the catalog level.

In [ ]:
# Model registration happens in Lab 05 after wrapping as ChatAgent.
# The ChatAgent interface is required for Model Serving deployment.
print("See Lab 05 for model registration with ChatAgent.")


## Key Takeaways

| Concept | What You Did |
|---|---|
| **RAG Pipeline** | Connected Vector Search retrieval to an LLM via a LangChain tool |
| **Tool Wrapping** | Decorated a plain Python function with `@tool` to make it agent-callable |
| **MLflow Logging** | Captured agent, signature, and parameters in a versioned MLflow run |
| **UC Registration** | Promoted the logged model to `{CATALOG}.{SCHEMA}.arxiv_research_agent` |

### Exam Tips

- The RAG pipeline order is always: **retrieve → augment prompt → generate**.  
- Use an **agent** (not a simple chain) when the model must decide *whether* and *when* to retrieve.  
- `infer_signature` captures input/output schemas — required for Model Serving validation.  
- `mlflow.set_registry_uri("databricks-uc")` must be called *before* `register_model` to target Unity Catalog.

## Key Concepts

| Concept | Definition |
|---|---|
| **RAG (Retrieval-Augmented Generation)** | Architecture that retrieves relevant documents and injects them into the prompt before generation, grounding the LLM in external knowledge |
| **Retrieval Tool** | A Python function decorated with `@tool` that the LangChain agent can invoke to fetch context from a data source |
| **ChatDatabricks** | LangChain-compatible wrapper for Databricks Foundation Model API endpoints (e.g., Llama-3.3-70B) |
| **ReAct Agent** | Agent created via `create_react_agent` (LangGraph) — the LLM uses function-calling to select and invoke tools dynamically; replaces the deprecated `create_tool_calling_agent` + `AgentExecutor` pattern |
| **LangGraph ReAct loop** | LangGraph manages the agent loop internally: observe → think → act → repeat until a final answer is produced; no separate `AgentExecutor` required |
| **MLflow Model Signature** | Schema object (`infer_signature`) that defines expected input columns/types and output format; enforced at serving time |
| **Unity Catalog Model Registry** | Databricks-native model registry that integrates with UC governance; accessed via `mlflow.set_registry_uri("databricks-uc")` |

## Exam Preparation

### Exam Practice Questions

**Q1.** What is the correct order of steps in a RAG pipeline?

- A) Generate → Retrieve → Augment
- B) Retrieve → Augment → Generate
- C) Augment → Retrieve → Generate
- D) Generate → Augment → Retrieve

**Answer: B** — Retrieve relevant documents first, augment the prompt with that context, then generate the response.

---

**Q2.** When should you use a LangChain **agent** instead of a simple **chain** for RAG?

- A) When retrieval always happens exactly once per query
- B) When the model should decide dynamically whether and how many times to retrieve
- C) When you want faster inference with fewer API calls
- D) When the prompt template has no placeholders

**Answer: B** — Agents are appropriate when the retrieval decision is conditional or when multi-hop retrieval may be needed.

---

**Q3.** Which MLflow function captures the input and output schema of a model for use with Model Serving?

- A) `mlflow.log_params()`
- B) `mlflow.langchain.log_model()`
- C) `infer_signature(model_input, model_output)`
- D) `mlflow.set_experiment()`

**Answer: C** — `infer_signature` inspects a sample input/output pair and returns a `ModelSignature` object that is passed to `log_model`.

---

**Q4.** What must you call before `mlflow.register_model()` to ensure the model is registered in Unity Catalog rather than the legacy Workspace Registry?

- A) `mlflow.set_tracking_uri("databricks")`
- B) `mlflow.set_registry_uri("databricks-uc")`
- C) `mlflow.enable_system_metrics_logging()`
- D) `mlflow.autolog()`

**Answer: B** — `mlflow.set_registry_uri("databricks-uc")` redirects `register_model` to Unity Catalog.

---

**Q5.** What is the purpose of the **system prompt** in the RAG agent configuration?

- A) To define the tool schemas the agent can call
- B) To set the maximum number of retrieval results
- C) To instruct the model on its persona, constraints, and when to use the retrieval tool
- D) To configure the MLflow experiment name

**Answer: C** — The system prompt shapes the agent's behaviour: persona ("arXiv Research Assistant"), constraints ("always retrieve before answering"), and citation format.

## Cost Breakdown

| Component | Detail | Estimated Cost |
|---|---|---|
| Databricks Serverless compute | Cluster startup + notebook execution (~20 min DBU) | ~$0.50 |
| LLM token usage | Test queries + sample inference for MLflow signature | ~$0.50 |
| Vector Search queries | Similarity search calls during testing and logging | Included in serverless |
| **Total** | | **~$1–2** |

**Estimated time:** ~30 min | **Estimated cost:** ~$1–2

> Costs vary by workspace region and current DBU pricing. Use the Databricks Cost Dashboard to track actuals.